In [58]:
from dataclasses import dataclass, field
from enum import StrEnum
from typing import Any
from broskill import SkillControl, ToolControl, Tool, Arg
from broskill.processing.tool import to_args
from broagent.codeblock import parse_json_codeblock
from broflow import BaseTask, TaskRegistry, Flow
import boto3
from brollm import BaseContract
from typing import Any, Callable
from pathlib import Path
import subprocess
import sys
from functools import partial
import yaml

ROOT = Path.cwd().resolve().parent
SKILL_DIR = ROOT / "skills"

sc = SkillControl(SKILL_DIR)
tc = ToolControl(sc)
sc.list_skills()


[Skill(name='ask-followup-question', description="Ask the user a clarifying question when you don't have enough information to proceed safely or correctly. Use only when something is genuinely missing or ambiguous -- never speculatively, and never for something you could reasonably infer from context already given.", version='v0.1.0', path=WindowsPath('D:/study-on-agent/skills/ask-followup-question'), tags=['meta', 'clarification'], keywords=None, default=True, status='experiment'),
 Skill(name='read-file', description="Read the contents of a specific file, or list files matching a pattern. Use when the user wants to see what's in a file, or wants to find files matching a pattern.", version='v0.1.0', path=WindowsPath('D:/study-on-agent/skills/read-file'), tags=['filesystem'], keywords=None, default=False, status='experiment'),
 Skill(name='skill-call', description="Gives skill-selection ability to models that don't natively support tool-calling, by defining a strict prompt-and-parse co

In [53]:
MODEL_LIST = [
    "google.gemma-3-4b-it",
    "google.gemma-3-12b-it", # support tool use
    "google.gemma-3-27b-it"
]

In [54]:
def pack_arg(arg:Arg):
    _base = {"type": arg.type}
    if arg.description:
        _base["description"] = arg.description
    return _base

def tool_to_yaml(tool:Tool):
    fn_metadata = dict(
        name=tool.name,
        description=tool.description,
        parameters={
            "type": "object",
            "properties": {arg.name: pack_arg(arg) for arg in tool.args},
            "required": [arg.name for arg in tool.args if arg.required==True]
        }
    )
    return {"type": "function", "function": fn_metadata}

In [55]:
# def register_tool(tool) -> dict:
#     """broskill Tool -> Bedrock toolSpec (different shape from OpenAI/Ollama's
#     {"type": "function", "function": {...}} -- Bedrock nests the JSON schema
#     one level deeper, under inputSchema.json)."""
#     return {
#         "toolSpec": {
#             "name": tool.name,
#             "description": tool.description,
#             "inputSchema": {
#                 "json": {
#                     "type": "object",
#                     "properties": {
#                         arg.name: {"type": arg.type, "description": arg.description}
#                         for arg in tool.args
#                     },
#                     "required": [arg.name for arg in tool.args if arg.required],
#                 }
#             },
#         }
#     }

In [56]:
class Process(StrEnum):
    SKILL_CALL = 'skill_call'
    TOOL_CALL = 'tool_call'
    TOOL_USE = 'tool_use'
    ASK_USER_QUESTION = 'ask_user_question'
    FAIL_RECOVERY = 'fail_recovery'
    ANSWER = 'answer'
    END = 'end'

MAX_RETRIES = 3

In [152]:
model = boto3.client('bedrock-runtime', region_name='us-east-1')

def UserMessage(text:str): return {'role': 'user', 'content': [{'text': text}]}
def AIMessage(text:Any): return text
def SystemMessage(text:str): return [{"text": text}]

def input_fn(
        messages:list[Any],
        system_prompt:Any|None=None,
        modelId:str|None=None,
)->Any:
    kwargs = {
        "modelId": modelId,
        "messages": messages,
    }
    if system_prompt:
        kwargs["system"] = system_prompt
    return model.converse(**kwargs)

def output_fn(response:Any)->Any:
    return response['output']['message']
    
gemma = BaseContract(input_fn=partial(input_fn, modelId="google.gemma-3-12b-it"), output_fn=output_fn)
gemma

In [153]:
messages = [UserMessage("Hello world")]
response = gemma(messages=messages, system_prompt=SystemMessage("you are a helpful agent"))

In [154]:
AIMessage(response)

{'role': 'assistant',
 'content': [{'text': "Hello to you too! 😊 \n\nHow can I be helpful today? Do you have any questions, tasks, or information you'd like me to assist with?"}]}

In [155]:
@dataclass
class State:
    messages:list
    skill_control:SkillControl
    tool_control:ToolControl
    system_prompt:str
    candidated_skills:list[Any] = field(default_factory=list)
    error_message:str = ''
    return_to:Process|None = None
    # input:str
    # skill_names:list = field(default_factory=list)
    # available_tools:list = field(default_factory=lambda: ['load_skill_extension', 'ask_user_question'])
    # skill_call_plan:list = field(default_factory=list)
    # tool_call_plan:list = field(default_factory=list)
    # reply_plan:list = field(default_factory=list)

    # return_to:Any = None
    # ask_user_question:str = ''
    # tool_calls:list = field(default_factory=list)
    # tool_results:list = field(default_factory=list)
    # error_message:str = ''
    # retry_count:int = 0
    # answer:str = ''

In [156]:
load_skill_tool = Tool(
    name="load_skill",
    description="Load the full instructions for a registered skill by name. Call this only when the current task clearly matches that skill's description.",
    args=[
        Arg(name="skill_name", type="string", required=True)
    ],
    path=Path()
)

load_skill_extension_tool = Tool(
    name="load_skill_extension",
    description="Load the full contents of one reference/asset document belonging to an already-loaded skill. Call this only when that skill's instructions point you to a specific reference/asset file for more detail — don't call it speculatively.",
    args=[
        Arg(name="skill_name", type="string", required=True, description="The name of the skill whose reference/asset you want to load."),
        Arg(name="path", type="string", required=True, description="The reference/asset file's path exactly as shown in the skill's instructions, e.g. `references/aws.md` or `assets/color.ts`.")
    ],
    path=Path()
)

ask_user_question_tool = Tool(
    name="ask_user_question",
    description="Signal that you need the user to answer something before you can continue. Write the actual question as your normal response content, then call this tool with no arguments to pause the turn and wait for their reply.",
    args=[
        Arg(name='question', type='string', required=True, description='questions you need user to clarify')
    ],
    path=Path()
)
tools = [
   tool_to_yaml(t)
   for t in [
      load_skill_tool,
      load_skill_extension_tool,
      ask_user_question_tool
   ]
]

available_tools = yaml.dump(tools, sort_keys=False)
print(available_tools)

- type: function
  function:
    name: load_skill
    description: Load the full instructions for a registered skill by name. Call this
      only when the current task clearly matches that skill's description.
    parameters:
      type: object
      properties:
        skill_name:
          type: string
      required:
      - skill_name
- type: function
  function:
    name: load_skill_extension
    description: "Load the full contents of one reference/asset document belonging\
      \ to an already-loaded skill. Call this only when that skill's instructions\
      \ point you to a specific reference/asset file for more detail \u2014 don't\
      \ call it speculatively."
    parameters:
      type: object
      properties:
        skill_name:
          type: string
          description: The name of the skill whose reference/asset you want to load.
        path:
          type: string
          description: The reference/asset file's path exactly as shown in the skill's
           

In [157]:
sc.list_skills()

[Skill(name='ask-followup-question', description="Ask the user a clarifying question when you don't have enough information to proceed safely or correctly. Use only when something is genuinely missing or ambiguous -- never speculatively, and never for something you could reasonably infer from context already given.", version='v0.1.0', path=WindowsPath('D:/study-on-agent/skills/ask-followup-question'), tags=['meta', 'clarification'], keywords=None, default=True, status='experiment'),
 Skill(name='read-file', description="Read the contents of a specific file, or list files matching a pattern. Use when the user wants to see what's in a file, or wants to find files matching a pattern.", version='v0.1.0', path=WindowsPath('D:/study-on-agent/skills/read-file'), tags=['filesystem'], keywords=None, default=False, status='experiment'),
 Skill(name='skill-call', description="Gives skill-selection ability to models that don't natively support tool-calling, by defining a strict prompt-and-parse co

In [ ]:
class SkillCall(BaseTask):
    possible_next = {Process.TOOL_CALL, Process.ANSWER, Process.FAIL_RECOVERY, Process.ASK_USER_QUESTION}
    def __init__(self, name, llm, system_prompt:str):
        super().__init__(name=name)
        self.llm = llm
        self.system_prompt = system_prompt

    def __call__(self, state:State)->State:
        try:
            skill_prompt = state.skill_control.load_skill('skill-call')
            available_skills = [f"- {s.name}: {s.description}" for s in state.skill_control.list_skills() if s.name not in ['skill-call', 'tool-call']]
            available_skills = f"## Available Skills:\n{'\n'.join(available_skills)}"
            available_tools = [
                tool_to_yaml(t) 
                for t in [
                    load_skill_tool, 
                    # load_skill_extension_tool, 
                    ask_user_question_tool
                ]
            ]
            available_tools = f"## Availale Tools:\n{yaml.dump(available_tools, sort_keys=False)}"
            local_prompt = f"{state.system_prompt}\n{self.system_prompt}\n{skill_prompt}\n{available_skills}\n{available_tools}"
            response = self.llm(state.messages, system_prompt=SystemMessage(local_prompt))
            candidated_skills = parse_json_codeblock(state.messages[-1]['content'][0]['text']).get('tool_use', [])
            if candidated_skills:
                self.set_next(Process.TOOL_USE)
                self.return_to = Process.TOOL_CALL
            else:
                self.set_next(Process.ANSWER)
            state.candidated_skills.extend(candidated_skills)
            state.messages.append(response)
        except Exception as e:
            state.error_message = str(e)
            state.return_to = Process.SKILL_CALL
            self.set_next(Process.FAIL_RECOVERY)
        return state

skill_call = SkillCall(name='skill-call', llm=gemma, system_prompt="You are an expert AI assistant who is good at identifying the correct skill to use for a given task.")
skill_call

In [163]:
system_prompt = ""
# content = "Hello World!"
# content = "What's up bro!"
# content = "I'm so blue today..."
# content = "Tell me a joke!"
# content = "What files are under skills directory?"
# content = "Read tell-joke/SKILL.md for me."
# content = "Read ./tell-joke/SKILL.md for me."
content = "List all files under skills directory and tell me some jokes."
# content = "List all files under skills directory and I'm so blue today..."
# content = "I'm so blue today, so list all files under skills directory."
state = State(
    messages=[UserMessage(content)],
    skill_control=sc,
    tool_control=tc,
    system_prompt=system_prompt
)
state = skill_call(state)

In [ ]:
# gemma 12b works, 4b fails
parse_json_codeblock(state.messages[-1]['content'][0]['text']).get('tool_use', [])

[{'name': 'load_skill', 'input': {'skill_name': 'read-file'}},
 {'name': 'load_skill', 'input': {'skill_name': 'tell-joke'}}]

In [137]:
state.candidated_skills

['read-file']

In [10]:
class AskUserQuestion(BaseTask):
    possible_next = {Process.SKILL_CALL, Process.TOOL_CALL}
    def __call__(self, state:State)->State:
        state.input = input(state.ask_user_question)
        self.set_next(state.return_to)
        return state

class SkillCall(BaseTask):
    """use small model as router"""
    possible_next = {Process.TOOL_CALL, Process.ANSWER, Process.FAIL_RECOVERY, Process.ASK_USER_QUESTION}
    def __call__(self, state:State)->State:
        return state

class ToolCall(BaseTask):
    """use small model as router"""
    possible_next = {Process.TOOL_CALL, Process.ANSWER, Process.FAIL_RECOVERY, Process.ASK_USER_QUESTION}
    def __call__(self, state:State)->State:
        return state

class ToolUse(BaseTask):
    possible_next = {Process.TOOL_CALL, Process.ANSWER, Process.FAIL_RECOVERY, Process.ASK_USER_QUESTION}
    def __call__(self, state:State)->State:
        return state

class Answer(BaseTask):
    possible_next = {Process.END}
    def __call__(self, state:State)->State:
        self.set_next(Process.END)
        return state

class FailRecovery(BaseTask):
    possible_next = {Process.SKILL_CALL, Process.TOOL_CALL, Process.ANSWER, Process.ASK_USER_QUESTION}
    def __call__(self, state:State)->State:
        self.set_next(state.return_to)
        return state